In [2]:
import numpy as np
import glob
import plotly.graph_objs as go
from plotly.offline import plot
import pyModeS as pms  # FIX: was never imported but used at the bottom in decode_mode_s()

# Decimation factor
# FIX: was defined twice — here as 1, then again inside synchronize_data() as 5
# (but placed after a return, so it was dead code). Consolidated here as 5 to
# match the apparent intent of the plotting code that uses data_decimate.
decimation_factor = 5

# Mode-S frame length in bits
mode_s_frame_length_bits = 112

# Samples per bit for Manchester decoding
sample_rate = 5e6  # 5 Msps
bit_rate = 1e6     # 1 Mbaud
samples_per_bit = int(sample_rate / bit_rate)

# Calculate the number of samples per Mode-S frame
samples_per_frame = mode_s_frame_length_bits * samples_per_bit


def synchronize_data(data):
    """
    Check the first 8 bits of the data for one of two known preamble patterns
    (p1 or p2) and return the data aligned so it starts with p1.
    If p2 is detected, prepend a synthetic '1' sample to shift the alignment.
    """
    data = np.asarray(data)
    # Binarise: positive samples → 1, others → 0
    bits = (data > 0).astype(np.uint8)

    p1 = np.array([1, 1, 1, 0, 0, 1, 1, 1], dtype=np.uint8)
    p2 = np.array([1, 1, 0, 0, 1, 1, 1, 0], dtype=np.uint8)

    if bits.size >= 8 and np.array_equal(bits[:8], p2):
        # p2 detected: prepend one sample so the frame lines up as p1
        out = np.concatenate((np.array([1.0], dtype=np.float32),
                              data.astype(np.float32)))
        return out
    elif bits.size >= 8 and np.array_equal(bits[:8], p1):
        return data.astype(np.float32)
    else:
        # Unknown preamble — return data unchanged
        return data.astype(np.float32)

    # FIX: everything below this point in the original was DEAD CODE — it was
    # placed after all return statements inside the function, so it never ran.
    # The decimation logic and the second trace-building block have been moved
    # to the main plotting loop below where they actually belong.


def decode_mode_s(data):
    """
    Decode a raw Mode-S frame (numpy array of float samples) into a
    human-readable message string using pyModeS, or return None if the
    frame is not decodable.
    """
    messages = []  # FIX: list is created but never used; kept for potential future use
    # Skip the first 8 samples used for preamble synchronization
    data = np.asarray(data[8:120])
    bits = (data > 0.5).astype(np.uint8)

    # Reshape into groups of 4 bits (nibbles) and convert each to a hex digit
    nibbles = bits.reshape(-1, 4)          # FIX: was .tolist() before multiply — that
    weights = np.array([8, 4, 2, 1],       # would fail because you can't multiply a
                       dtype=np.uint8)     # plain Python list by a numpy array element-wise
    vals = (nibbles * weights).sum(axis=1)  # weighted sum → value 0-15 per nibble

    # Map each nibble value to its ASCII hex character via a lookup table
    lut = np.frombuffer(b"0123456789ABCDEF", dtype="S1")
    hex_chars = lut[vals].astype(str)
    hex_str = "".join(hex_chars.tolist())

    message = pms.tell(hex_str)
    if message is not None:
        return message
    # FIX: added an implicit return None for the case where pms.tell() returns None
    return None


# Search for .dat files in the current directory
data_files = glob.glob("*.dat")

# Build Plotly traces for each data file
traces = []
for file in data_files:
    data = np.fromfile(open(file), dtype=np.float32)

    # Skip files that are too short to contain a full Mode-S frame
    if len(data) >= samples_per_frame:
        # FIX: original code referenced 'data_decimate' here before it was ever
        # defined, causing a NameError. The synchronization + decimation steps
        # that produce data_decimate have been moved here from inside
        # synchronize_data() where they were unreachable dead code.
        data_sync = synchronize_data(data)
        data_decimate = data_sync[1::decimation_factor]  # downsample by decimation_factor

        trace = go.Scatter(
            x=np.linspace(0, len(data_decimate) - 1, len(data_decimate)),
            y=data_decimate,
            mode='lines+markers',
            name=file
        )
        traces.append(trace)

# Create layout for the plot
layout = go.Layout(
    title='Data from .dat files',
    xaxis=dict(title='Sample index'),
    yaxis=dict(title='Amplitude'),
)

fig = go.Figure(data=traces, layout=layout)

# Save the interactive plot as an HTML file
plot(fig, filename='data_plot.html')

ModuleNotFoundError: No module named 'plotly'